# Mã hóa Ladder AMK (At-Most-K) cho Sequence Constraints

## 1. Cơ sở lý thuyết

Bài toán Sequence Constraints yêu cầu: Trong một chuỗi $N$ biến số, bất kỳ cửa sổ trượt (sliding window) nào có kích thước $W$ cũng chỉ chứa tối đa $K$ biến mang giá trị `True`. 

Cách tiếp cận ngây thơ (Naive Sliding Window) là áp dụng bộ đếm AMK cho từng cửa sổ trượt một cách độc lập. Điều này sinh ra $N - W + 1$ bộ đếm, dẫn đến sự bùng nổ số lượng biến phụ và mệnh đề, đặc biệt do các cửa sổ liền kề giao nhau (overlap) tới $W-1$ biến.

**Ladder AMK Encoding** giải quyết sự dư thừa này bằng chiến lược phân rã thành các khối (Block-based decomposition):
1. Chia toàn bộ chuỗi $N$ biến thành các khối (Area/Block) rời rạc, mỗi khối có kích thước tối đa $W$.
2. Mã hóa giới hạn số lượng biến `True` bên trong nội bộ mỗi khối bằng Sequential Counter.
3. Để kiểm soát các cửa sổ trượt nằm vắt ngang giữa hai khối kề nhau (Khối $i$ và Khối $i+1$), Ladder AMK tạo ra các **mệnh đề liên kết (Connection clauses)**. Cụ thể, nó liên kết bộ đếm hậu tố (Suffix/Reverse counter) của khối $i$ với bộ đếm tiền tố (Prefix/Forward counter) của khối $i+1$. Nếu hậu tố của khối $i$ có $P$ biến `True`, thì tiền tố của khối $i+1$ được phép có tối đa $K - P$ biến `True`.

## 2. Thuật toán và các bước thực hiện

**Bước 1: Chia khối (Area Partitioning)**
Chuỗi đầu vào được cắt thành các mảng con kích thước $W$. Giả sử $N=10, W=4$, ta có 3 khối: $A_0$ (biến 1-4), $A_1$ (biến 5-8), $A_2$ (biến 9-10).

**Bước 2: Xây dựng bộ đếm Sequential Counter (SC)**
Tạo hàm SC lõi để ánh xạ biến đầu vào $X$ thành ma trận trạng thái $R_{i,j}$ ($R_{i,j}$ mang giá trị `True` nếu mảng con từ vị trí $1$ đến $i$ có ít nhất $j$ biến `True`). Hàm này cần hỗ trợ 2 chế độ:
- Chế độ "Đếm và Ràng buộc" (AMK Block): Áp dụng cho mảng thuận. Bật tính năng chặn nếu số lượng vượt quá $K$.
- Chế độ "Chỉ Đếm" (Connection Block): Áp dụng cho mảng bị đảo ngược (hậu tố). Không ép ràng buộc vượt $K$, chỉ dùng để trích xuất trạng thái kết nối.

**Bước 3: Liên kết ranh giới (Cross-boundary Linking)**
Giữa khối $A_i$ (đã đếm ngược) và khối $A_{i+1}$ (đã đếm thuận), có $W-1$ cửa sổ trượt vắt ngang. Ta duyệt qua các tổ hợp độ dài $j_{left} + j_{right} = W$ và chèn mệnh đề loại trừ mâu thuẫn: $\neg (R_{left} \ge K - P + 1 \land R_{right} \ge P)$.

## 3. Triển khai mã nguồn

### 3.1. Lõi đếm tuần tự (Sequential Counter Core)

In [ ]:
def create_counter(solver, block, k, start_id, enforce_k):
    n = len(block)
    R = {}
    curr_id = start_id

    for i in range(1, n + 1):
        R[i] = {}
        for j in range(1, min(i, k) + 1):
            R[i][j] = curr_id
            curr_id += 1

    for i in range(1, n):
        solver.add_clause([-block[i-1], R[i][1]])

    for i in range(2, n + 1):
        for j in range(1, min(i-1, k) + 1):
            solver.add_clause([-R[i-1][j], R[i][j]])

    for i in range(2, n + 1):
        for j in range(2, min(i, k) + 1):
            solver.add_clause([-block[i-1], -R[i-1][j-1], R[i][j]])

    for i in range(1, min(n, k) + 1):
        solver.add_clause([block[i-1], -R[i][i]])

    for i in range(2, n + 1):
        for j in range(2, min(i, k) + 1):
            solver.add_clause([R[i-1][j-1], -R[i][j]])

    for i in range(2, n + 1):
        for j in range(1, min(i-1, k) + 1):
            solver.add_clause([block[i-1], R[i-1][j], -R[i][j]])

    if enforce_k:
        for i in range(k + 1, n + 1):
            solver.add_clause([-block[i-1], -R[i-1][k]])

    return R, curr_id

**Giải thích hàm `create_counter`:**
Hàm sinh ra tập mệnh đề đảm bảo tính toàn vẹn của ma trận đếm $R$. Có 7 nhóm mệnh đề cốt lõi tương ứng với 7 vòng lặp:
1. $x_i \to R_{i,1}$: Nếu phần tử hiện tại đúng, mảng có ít nhất 1 giá trị đúng.
2. $R_{i-1, s} \to R_{i, s}$: Tính kế thừa. Nếu trước đó đã có $s$ giá trị đúng, hiện tại cũng có $s$ giá trị đúng.
3. $x_i \land R_{i-1, s-1} \to R_{i, s}$: Nếu trước đó có $s-1$ giá trị đúng và phần tử hiện tại đúng, tổng số tăng lên $s$.
4. $\neg x_i \to \neg R_{i,i}$: Nếu phần tử hiện tại sai, không thể đạt số lượng đúng tối đa tuyệt đối là $i$.
5. $\neg R_{i-1, s-1} \to \neg R_{i, s}$: Nếu trước đó không đạt $s-1$, thì việc cộng thêm phần tử hiện tại không thể giúp đạt $s$.
6. $\neg x_i \land \neg R_{i-1, s} \to \neg R_{i, s}$: Nếu trước đó không đạt $s$ và phần tử hiện tại sai, tổng cũng không thể đạt $s$.
7. $x_i \to \neg R_{i-1, k}$: (Chỉ chạy khi `enforce_k = True`). Ràng buộc AMK chính: Không cho phép phần tử thứ $i$ mang giá trị `True` nếu trước đó đã tích lũy đủ $K$ phần tử `True`.

---

### 3.2. Cấu trúc liên kết ranh giới

In [ ]:
def link_blocks(solver, R_left, R_right, overlap, k):
    for i in range(overlap):
        j_left = i + 1
        j_right = overlap - i
        for p in range(1, k + 1):
            if k - p + 1 <= j_left and p <= j_right:
                if (k - p + 1) in R_left[j_left] and p in R_right[j_right]:
                    left_var = R_left[j_left][k - p + 1]
                    right_var = R_right[j_right][p]
                    solver.add_clause([-left_var, -right_var])

**Giải thích hàm `link_blocks`:**
Cửa sổ trượt nằm đè lên hai khối luôn có tổng số lượng biến bằng với $W$. Nếu $j_{left}$ là số biến thuộc khối bên trái, thì phần còn lại $j_{right} = W - j_{left}$ sẽ thuộc khối bên phải. Hàm duyệt qua tất cả khả năng phân bổ số biến `True` (giá trị $P$) giữa hai phần này. Nếu phần bên phải có $P$ biến `True`, phần bên trái không được phép có $\ge K - P + 1$ biến `True`. Mệnh đề `[-left_var, -right_var]` thiết lập ranh giới xung đột này.

---

### 3.3. Tích hợp chuỗi Ladder AMK

In [ ]:
import math

def ladder_amk_sequence(solver, variables, w, k, start_id):
    n = len(variables)
    next_id = start_id
    areas = math.ceil(n / w)

    blocks_forward = {}
    blocks_backward = {}

    for i in range(areas):
        start_idx = i * w
        end_idx = min((i + 1) * w, n)
        chunk = variables[start_idx:end_idx]

        if i != 0:
            R_fwd, next_id = create_counter(solver, chunk, k, next_id, True)
            blocks_forward[i] = R_fwd

        if i != areas - 1:
            R_bwd, next_id = create_counter(solver, chunk[::-1], k, next_id, False)
            blocks_backward[i] = R_bwd

    for i in range(areas - 1):
        if i in blocks_backward and (i + 1) in blocks_forward:
            link_blocks(solver, blocks_backward[i], blocks_forward[i+1], w - 1, k)

    return next_id

**Giải thích hàm `ladder_amk_sequence`:**
1. Cắt chuỗi `variables` thành các mảng con (chunk) độ dài $W$.
2. Nếu không phải khối đầu tiên, gọi bộ đếm thuận với `enforce_k = True`. Các khối thuận này đại diện cho nửa sau của một chuỗi liên kết.
3. Nếu không phải khối cuối cùng, mảng con được đảo ngược (reverse) và gọi bộ đếm với `enforce_k = False`. Các khối nghịch này làm nửa trước (hậu tố) để chuẩn bị nối vào nửa sau của khối kế tiếp.
4. Vòng lặp cuối thực hiện ghép cặp mảng hậu tố của khối $i$ và mảng tiền tố của khối $i+1$ qua hàm `link_blocks`.

---

### 3.4. Mô phỏng thực thi

In [ ]:
from pysat.solvers import Glucose3

def demo_amk():
    solver = Glucose3()
    n = 10
    w = 4
    k = 2

    variables = list(range(1, n + 1))

    ladder_amk_sequence(solver, variables, w, k, start_id=11)

    # Giả lập vi phạm: Gán 3 biến liên tiếp bằng True
    solver.add_clause([3])
    solver.add_clause([4])
    solver.add_clause([5])

    if solver.solve():
        print("SAT - Tìm được nghiệm!")
    else:
        print("UNSAT - Không có nghiệm (Đã bị chặn đúng như lý thuyết)!")

    solver.delete()

demo_amk()

**Giải thích mã mô phỏng:**
Kịch bản thử nghiệm yêu cầu mọi cửa sổ kích thước $W=4$ chỉ có tối đa $K=2$ biến `True`. Chúng ta chủ động gán các biến $x_3, x_4, x_5$ đều bằng `True`. 
Khi cửa sổ trượt dịch chuyển tới tọa độ bao trọn 3 biến này (ví dụ cửa sổ gồm các biến $\{2, 3, 4, 5\}$ hoặc $\{3, 4, 5, 6\}$), hệ thống sẽ phát hiện số biến `True` là $3$, vi phạm ràng buộc $K \le 2$. Bộ giải sẽ trả về trạng thái **UNSAT** chính xác như kỳ vọng lý thuyết.